# Live Data Processing - Google Colab

## Live Data Processing and Dashboard Updates

This Colab notebook processes live data and updates dashboards in real-time.

In [ ]:
# Installation
!pip install pandas numpy matplotlib seaborn plotly dash
!pip install google-cloud-bigquery azure-storage-blob
!pip install streamlit gradio

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from datetime import datetime, timedelta
import time

# Style settings
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Live data connection setup

class LiveDataProcessor:
    def __init__(self):
        self.data_sources = {
            'bigquery': 'https://your-bigquery-endpoint',
            'azure': 'https://your-azure-storage',
            'api': 'https://your-data-api.com'
        }
    
    def fetch_live_data(self, source='sample'):
        """Fetch live data"""
        if source == 'sample':
            # Generate sample live data
            np.random.seed(int(time.time()))
            data = {
                'timestamp': pd.date_range(start='2024-01-01', periods=100, freq='H'),
                'users': np.random.randint(50, 500, 100),
                'revenue': np.random.uniform(1000, 10000, 100),
                'page_views': np.random.randint(100, 1000, 100),
                'conversion_rate': np.random.uniform(0.01, 0.05, 100)
            }
            return pd.DataFrame(data)
        else:
            # Real API call
            # TODO: Real API integration
            pass
    
    def process_data(self, df):
        """Data processing"""
        # Real-time metrics
        metrics = {
            'total_users': df['users'].sum(),
            'total_revenue': df['revenue'].sum(),
            'avg_conversion': df['conversion_rate'].mean(),
            'peak_hour': df.loc[df['users'].idxmax(), 'timestamp'].hour,
            'last_updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        return df, metrics

processor = LiveDataProcessor()

In [ ]:
# Interactive dashboard

def create_dashboard(df, metrics):
    """Create interactive dashboard"""
    
    # Subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Users Trend', 'Revenue', 'Conversion Rate', 'Page Views'),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # Users trend
    fig.add_trace(
        go.Scatter(x=df['timestamp'], y=df['users'], name='Users', line=dict(color='blue')),
        row=1, col=1
    )
    
    # Revenue
    fig.add_trace(
        go.Scatter(x=df['timestamp'], y=df['revenue'], name='Revenue', line=dict(color='green')),
        row=1, col=2
    )
    
    # Conversion rate
    fig.add_trace(
        go.Scatter(x=df['timestamp'], y=df['conversion_rate']*100, name='Conversion %', line=dict(color='red')),
        row=2, col=1
    )
    
    # Page views
    fig.add_trace(
        go.Scatter(x=df['timestamp'], y=df['page_views'], name='Page Views', line=dict(color='purple')),
        row=2, col=2
    )
    
    fig.update_layout(
        title=f'Live Dashboard - Updated: {metrics["last_updated"]}',
        height=800,
        showlegend=True
    )
    
    return fig

# Display dashboard
df, metrics = processor.process_data(processor.fetch_live_data())
dashboard_fig = create_dashboard(df, metrics)
dashboard_fig.show()

In [ ]:
# Real-time update function

def update_dashboard(interval=30):
    """Update dashboard at given interval"""
    
    while True:
        try:
            # Get new data
            new_data = processor.fetch_live_data()
            df, metrics = processor.process_data(new_data)
            
            # Update dashboard
            fig = create_dashboard(df, metrics)
            fig.show()
            
            # Print metrics
            print(f"\nMetrics Update - {metrics['last_updated']}")
            print(f"Total Users: {metrics['total_users']:,}")
            print(f"Total Revenue: ${metrics['total_revenue']:,.2f}")
            print(f"Avg Conversion: {metrics['avg_conversion']*100:.2f}%")
            print(f"Peak Hour: {metrics['peak_hour']}:00")
            print("-" * 50)
            
            # Wait
            time.sleep(interval)
            
        except KeyboardInterrupt:
            print("\nDashboard updates stopped")
            break
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(5)

print("To start real-time updates, run:")
print("update_dashboard(interval=30)  # Update every 30 seconds")

In [ ]:
# Mobile friendly dashboard (Streamlit style)

def create_mobile_dashboard(df, metrics):
    """Mobile friendly dashboard"""
    
    # Main metrics
    fig = go.Figure()
    
    fig.add_trace(go.Indicator(
        mode = "number+gauge+delta",
        value = metrics['total_users'],
        domain = {'x': [0, 0.5], 'y': [0, 1]},
        title = {'text': "Total Users"},
        delta = {'reference': metrics['total_users'] * 0.9},
        gauge = {
            'axis': {'range': [None, metrics['total_users'] * 1.2]},
            'bar': {'color': "darkblue"},
            'steps': [
                {'range': [0, metrics['total_users'] * 0.5], 'color': "lightgray"},
                {'range': [metrics['total_users'] * 0.5, metrics['total_users']], 'color': "gray"}
            ],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': metrics['total_users'] * 0.95
            }
        }
    ))
    
    fig.add_trace(go.Indicator(
        mode = "number+gauge+delta",
        value = metrics['total_revenue'],
        domain = {'x': [0.5, 1], 'y': [0, 1]},
        title = {'text': "Revenue ($)"},
        delta = {'reference': metrics['total_revenue'] * 0.9},
        gauge = {
            'axis': {'range': [None, metrics['total_revenue'] * 1.2]},
            'bar': {'color': "green"},
            'steps': [
                {'range': [0, metrics['total_revenue'] * 0.5], 'color': "lightgray"},
                {'range': [metrics['total_revenue'] * 0.5, metrics['total_revenue']], 'color': "gray"}
            ],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': metrics['total_revenue'] * 0.95
            }
        }
    ))
    
    fig.update_layout(
        title=f'Mobile Dashboard - {metrics["last_updated"]}',
        height=400
    )
    
    return fig

# Display mobile dashboard
mobile_fig = create_mobile_dashboard(df, metrics)
mobile_fig.show()

In [ ]:
# API endpoint creation

def create_api_endpoint():
    """Create API endpoint for dashboard"""
    
    # Data in JSON format
    api_data = {
        'metrics': metrics,
        'chart_data': {
            'timestamps': df['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S').tolist(),
            'users': df['users'].tolist(),
            'revenue': df['revenue'].tolist(),
            'conversion_rates': (df['conversion_rate'] * 100).tolist()
        },
        'last_updated': metrics['last_updated']
    }
    
    return json.dumps(api_data, indent=2, default=str)

# Display API response
api_response = create_api_endpoint()
print("API Response:")
print(api_response[:500] + "...")

## Updates Scheduling

### Automatic Updates:
- **Online**: Every 30 seconds
- **Offline**: Every 5 minutes  
- **Nightly**: Every 1 hour

### Data Sources:
- BigQuery (real-time)
- Azure Storage (batch updates)
- Custom APIs (live feeds)

### Dashboard Links:
- **Colab**: [Link] (real-time updates)
- **Looker Studio**: [Link] (static dashboard)
- **Azure Workbook**: [Link] (enterprise dashboard)